# AutoPipe — Uso básico

Este cuaderno muestra el flujo completo de **AutoPipe**: cargar un dataset de
imágenes, definir un espacio de preprocesamiento, buscar automáticamente el mejor
pipeline para un modelo de detección de anomalías y generar un reporte reproducible.

AutoPipe **no entrena el modelo base**: lo consume mediante la interfaz
`fit`/`predict` de scikit-learn. Aquí usamos `IsolationForest` como ejemplo.

In [1]:
import tempfile
from pathlib import Path

import numpy as np
from PIL import Image
from sklearn.ensemble import IsolationForest

from autopipe import (
    GaussianBlur,
    GridSearchEngine,
    ImageDataLoader,
    Normalize,
    OptunaSearchEngine,
    ParallelRunner,
    ReportGenerator,
    Resize,
)

C:\Users\ricar\OneDrive - Estudiantes ITCR\2026 - I Semestre\Proyecto de Aplicación de Ingeniería\AutoPipe\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Crear un dataset de ejemplo

AutoPipe espera un directorio con subcarpetas `normal/` y `anomaly/`. Aquí
generamos un dataset sintético donde las anomalías son notablemente más brillantes
y, sobre todo, **raras** (40 normales vs. 10 anómalas, 20 %): `IsolationForest`
—como la mayoría de los detectores de anomalías— asume que las anomalías son
minoritarias, y su umbral de decisión se degrada con clases balanceadas 50/50.

In [2]:
rng = np.random.default_rng(0)
dataset_dir = Path(tempfile.mkdtemp()) / 'dataset'

for label, (folder, low, high, count) in {
    0: ('normal', 0, 40, 40),
    1: ('anomaly', 200, 256, 10),
}.items():
    (dataset_dir / folder).mkdir(parents=True, exist_ok=True)
    for i in range(count):
        arr = rng.integers(low, high, size=(64, 64, 3), dtype=np.uint8)
        Image.fromarray(arr).save(dataset_dir / folder / f'img_{i:02d}.png')

print('Dataset creado en:', dataset_dir)

Dataset creado en: C:\Users\ricar\AppData\Local\Temp\tmpwe958vft\dataset


## 2. Cargar el dataset

In [3]:
images, labels = ImageDataLoader(dataset_path=dataset_dir).load()
print('Imágenes:', images.shape, '| Etiquetas:', labels.shape)
print('Normales:', int((labels == 0).sum()), '| Anomalías:', int((labels == 1).sum()))

Imágenes: (50, 64, 64, 3) | Etiquetas: (50,)
Normales: 40 | Anomalías: 10


## 3. Definir el espacio de búsqueda

Cada clave es un paso del pipeline y su valor una lista de transformaciones
candidatas. La búsqueda en malla prueba el producto cartesiano de todas ellas.

In [4]:
search_space = {
    'resize': [Resize(width=32, height=32)],
    'blur': [GaussianBlur(kernel_size=3), GaussianBlur(kernel_size=5)],
    'normalize': [Normalize(mean=0.5, std=0.5)],
}

## 4. Búsqueda en malla (Grid Search)

`run(...)` devuelve `(best_pipeline, best_metrics, results)`.

In [5]:
# contamination=0.2 le indica al modelo la proporción esperada de anomalías
# (10 de 50); así su umbral de predicción binaria queda bien calibrado.
base_model = IsolationForest(contamination=0.2, random_state=0)

engine = GridSearchEngine(search_space, val_size=0.3, random_state=42)
best_pipeline, metrics, results = engine.run(
    images, labels, base_model=base_model, metric='auc_roc',
)

print('Mejor pipeline:', best_pipeline)
print('Métricas del ganador:')
for name, value in metrics.items():
    print(f'  {name:>20}: {value:.4f}')

Mejor pipeline: Pipeline([Resize(width=32, height=32), GaussianBlur(kernel_size=3), Normalize(mean=0.5, std=0.5)])
Métricas del ganador:
               auc_roc: 1.0000
                    f1: 1.0000
             precision: 1.0000
                recall: 1.0000
   false_positive_rate: 0.0000


### Tabla comparativa de todos los pipelines evaluados

In [6]:
for record in results:
    print(f"#{record['pipeline_index']}  auc_roc={record['metrics']['auc_roc']:.4f}"
          f"  params={record['params']}")

#0  auc_roc=1.0000  params={'0_Resize': {'width': 32, 'height': 32}, '1_GaussianBlur': {'kernel_size': 3}, '2_Normalize': {'mean': 0.5, 'std': 0.5}}
#1  auc_roc=1.0000  params={'0_Resize': {'width': 32, 'height': 32}, '1_GaussianBlur': {'kernel_size': 3}}
#2  auc_roc=1.0000  params={'0_Resize': {'width': 32, 'height': 32}, '1_GaussianBlur': {'kernel_size': 5}, '2_Normalize': {'mean': 0.5, 'std': 0.5}}
#3  auc_roc=1.0000  params={'0_Resize': {'width': 32, 'height': 32}, '1_GaussianBlur': {'kernel_size': 5}}
#4  auc_roc=1.0000  params={'0_Resize': {'width': 32, 'height': 32}, '1_Normalize': {'mean': 0.5, 'std': 0.5}}
#5  auc_roc=1.0000  params={'0_Resize': {'width': 32, 'height': 32}}
#6  auc_roc=1.0000  params={'0_GaussianBlur': {'kernel_size': 3}, '1_Normalize': {'mean': 0.5, 'std': 0.5}}
#7  auc_roc=1.0000  params={'0_GaussianBlur': {'kernel_size': 3}}
#8  auc_roc=1.0000  params={'0_GaussianBlur': {'kernel_size': 5}, '1_Normalize': {'mean': 0.5, 'std': 0.5}}
#9  auc_roc=1.0000  params

## 5. Alternativa: búsqueda bayesiana con Optuna

`OptunaSearchEngine` acepta el mismo `search_space` y la misma firma de `run`.

In [7]:
optuna_engine = OptunaSearchEngine(search_space, n_trials=8, val_size=0.3, random_state=42)
opt_pipeline, opt_metrics, _ = optuna_engine.run(
    images, labels, base_model=IsolationForest(contamination=0.2, random_state=0),
    metric='auc_roc',
)
print('Mejor pipeline (Optuna):', opt_pipeline)
print('AUC-ROC:', round(opt_metrics['auc_roc'], 4))

[I 2026-07-01 16:05:59,229] A new study created in memory with name: no-name-827604db-b0f5-4765-b983-946d27d4099d


[I 2026-07-01 16:05:59,342] Trial 0 finished with value: 1.0 and parameters: {'resize': 1, 'blur': 0, 'normalize': 0}. Best is trial 0 with value: 1.0.


[I 2026-07-01 16:05:59,447] Trial 1 finished with value: 1.0 and parameters: {'resize': 0, 'blur': 2, 'normalize': 0}. Best is trial 0 with value: 1.0.


[I 2026-07-01 16:05:59,558] Trial 2 finished with value: 1.0 and parameters: {'resize': 1, 'blur': 1, 'normalize': 1}. Best is trial 0 with value: 1.0.


[I 2026-07-01 16:05:59,672] Trial 3 finished with value: 1.0 and parameters: {'resize': 1, 'blur': 2, 'normalize': 1}. Best is trial 0 with value: 1.0.


[I 2026-07-01 16:05:59,786] Trial 4 finished with value: 1.0 and parameters: {'resize': 0, 'blur': 0, 'normalize': 1}. Best is trial 0 with value: 1.0.


[I 2026-07-01 16:05:59,899] Trial 5 finished with value: 1.0 and parameters: {'resize': 0, 'blur': 1, 'normalize': 1}. Best is trial 0 with value: 1.0.


[I 2026-07-01 16:06:00,010] Trial 6 finished with value: 1.0 and parameters: {'resize': 1, 'blur': 1, 'normalize': 1}. Best is trial 0 with value: 1.0.


[I 2026-07-01 16:06:00,125] Trial 7 finished with value: 1.0 and parameters: {'resize': 1, 'blur': 1, 'normalize': 1}. Best is trial 0 with value: 1.0.


Mejor pipeline (Optuna): Pipeline([GaussianBlur(kernel_size=3), Normalize(mean=0.5, std=0.5)])
AUC-ROC: 1.0


## 6. Evaluación en paralelo y speedup

`ParallelRunner.benchmark` compara la ejecución secuencial y paralela y verifica
que producen el mismo resultado.

In [8]:
pipelines = engine.build_pipelines()
runner = ParallelRunner(n_jobs=-1, val_size=0.3, random_state=42)
bench = runner.benchmark(
    pipelines, images, labels,
    base_model=IsolationForest(contamination=0.2, random_state=0), metric='auc_roc',
)
print(f"Speedup: {bench['speedup']:.2f}x | consistente: {bench['consistent']}")

Speedup: 0.25x | consistente: True


## 7. Generar el reporte

El HTML se genera siempre; el PDF es opcional (requiere `wkhtmltopdf` en el PATH).
Sin `output_dir`, los archivos se escriben en una subcarpeta `reports/` **junto al
script o notebook que invoca la librería** (se crea automáticamente). Pasamos
también las imágenes y un puntaje de anomalía para destacar las más anómalas.

In [9]:
scores = images.reshape(len(images), -1).mean(axis=1)  # brillo medio como score de ejemplo

# Sin output_dir: escribe en reports/ junto a este notebook (se crea sola).
report = ReportGenerator(top_anomalies=5).generate(
    results, best_pipeline, metrics, metric='auc_roc', images=images, scores=scores,
)
print('HTML:', report['html'])
print('PDF :', report['pdf'])  # None si wkhtmltopdf no está disponible
print('Config ganadora:', report['config'])

HTML: C:\Users\ricar\OneDrive - Estudiantes ITCR\2026 - I Semestre\Proyecto de Aplicación de Ingeniería\AutoPipe\examples\reports\report.html
PDF : C:\Users\ricar\OneDrive - Estudiantes ITCR\2026 - I Semestre\Proyecto de Aplicación de Ingeniería\AutoPipe\examples\reports\report.pdf
Config ganadora: C:\Users\ricar\OneDrive - Estudiantes ITCR\2026 - I Semestre\Proyecto de Aplicación de Ingeniería\AutoPipe\examples\reports\winning_pipeline.json


## 8. Reproducir el pipeline ganador desde la configuración guardada

La configuración del ganador se puede recargar y reconstruir para re-ejecutar.

In [10]:
config = ReportGenerator.load_pipeline_config(report['config'])
rebuilt = ReportGenerator.rebuild_pipeline(config)
print('Reconstruido:', rebuilt)
print('Coincide con el ganador:', rebuilt.get_params() == best_pipeline.get_params())

Reconstruido: Pipeline([Resize(width=32, height=32), GaussianBlur(kernel_size=3), Normalize(mean=0.5, std=0.5)])
Coincide con el ganador: True
